In [9]:
import os
import pandas as pd
import numpy as np
import logging
import sys

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
log = logging.getLogger(__name__)

REQUIRED_COLUMNS = [
    'Call Id', 'Date', 'Agent', 'Agent Group',
    'Answered (Y/N)', 'Issue Resolved?', 'Resolved within SLA?',
    'First Call Resolution', 'Speed of Answer (Secs)',
    'Talk Time (Secs)', 'Hold Time (Secs)',
    'Satisfaction rating', 'Main Call Reason', 'Sub Call Reason'
]

YN_COLS = ['Answered (Y/N)', 'Issue Resolved?', 'Resolved within SLA?', 'First Call Resolution']

SATISFACTION_MAP = {
    'Extremely Dissatisfied': 1,
    'Dissatisfied':           2,
    'Neutral':                3,
    'Satisfied':              4,
    'Extremely Satisfied':    5
}


# =============================================================================
# VALIDATION
# =============================================================================

def validate_input(df):
    missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")
    log.info(f"Input validated — {len(df):,} rows, {len(df.columns)} columns")


def check_dim_uniqueness(dim_agent, dim_satisfaction, dim_reason):
    checks = [
        (dim_agent,        'Agent',                                 'Dim_Agent'),
        (dim_satisfaction, 'Satisfaction rating',                   'Dim_Satisfaction'),
        (dim_reason,       ['Main Call Reason', 'Sub Call Reason'], 'Dim_Reason'),
    ]
    for dim, key, name in checks:
        dupes = dim.duplicated(subset=key).sum()
        if dupes > 0:
            log.warning(f"{name}: {dupes} duplicate keys on '{key}' — will cause row multiplication on merge!")
        else:
            log.info(f"{name}: keys are unique ✓")


def validate_fact_table(fact, source_df):
    assert len(fact) == len(source_df), (
        f"Row count mismatch after merge! Fact={len(fact)}, Source={len(source_df)}. "
        "Check for duplicate keys in dimension tables."
    )
    nulls     = fact.isnull().sum()
    null_cols = nulls[nulls > 0]
    if not null_cols.empty:
        log.warning(f"Null values in fact table:\n{null_cols}")
    log.info(f"Fact_Table validated — {len(fact):,} rows, {fact.isnull().sum().sum()} nulls total")


# =============================================================================
# HELPERS
# =============================================================================

def convert_yn_columns(df):
    for col in YN_COLS:
        nulls      = df[col].isna().sum()
        unexpected = (~df[col].isin(['Y', 'N']) & df[col].notna()).sum()
        if nulls > 0:
            log.warning(f"'{col}': {nulls} null values → mapped to 0")
        if unexpected > 0:
            log.warning(f"'{col}': {unexpected} unexpected values → mapped to 0")
        df[col] = df[col].map({'Y': 1, 'N': 0}).fillna(0).astype(int)
    return df


# =============================================================================
# DIMENSION BUILDERS
# =============================================================================

def build_dim_agent_group(df):
    dim = (
        df[['Agent Group']]
        .drop_duplicates()
        .sort_values('Agent Group')
        .reset_index(drop=True)
    )
    dim.insert(0, 'Agent_Group_ID', range(1, len(dim) + 1))
    log.info(f"Dim_Agent_Group: {len(dim)} rows")
    return dim


def build_dim_agent(df, dim_agent_group):
    dim = (
        df[['Agent', 'Agent Group']]
        .drop_duplicates()
        .sort_values(['Agent Group', 'Agent'])
        .reset_index(drop=True)
    )
    dim = dim.merge(dim_agent_group, on='Agent Group', how='left')
    dim.insert(0, 'Agent_ID', range(1, len(dim) + 1))
    dim = dim[['Agent_ID', 'Agent', 'Agent_Group_ID']]
    log.info(f"Dim_Agent: {len(dim)} rows")
    return dim


def build_dim_satisfaction(df):
    unknown_row = pd.DataFrame([{
        'Satisfaction_rating_ID': 0,
        'Satisfaction rating':    'N/A — Call Not Answered',
        'Satisfaction_Score':     np.nan
    }])
    dim = (
        df[['Satisfaction rating']]
        .dropna()
        .drop_duplicates('Satisfaction rating')
        .sort_values('Satisfaction rating')
        .reset_index(drop=True)
    )
    dim.insert(0, 'Satisfaction_rating_ID', range(1, len(dim) + 1))
    dim['Satisfaction_Score'] = dim['Satisfaction rating'].map(SATISFACTION_MAP)
    dim = pd.concat([unknown_row, dim], ignore_index=True)
    log.info(f"Dim_Satisfaction: {len(dim)} rows (incl. N/A placeholder)")
    return dim


def build_dim_reason(df):
    unknown_row = pd.DataFrame([{
        'Reason_ID':        0,
        'Main Call Reason': 'N/A — Call Not Answered',
        'Sub Call Reason':  'N/A — Call Not Answered'
    }])
    dim = (
        df[['Main Call Reason', 'Sub Call Reason']]
        .dropna()
        .drop_duplicates(['Main Call Reason', 'Sub Call Reason'])
        .sort_values(['Main Call Reason', 'Sub Call Reason'])
        .reset_index(drop=True)
    )
    dim.insert(0, 'Reason_ID', range(1, len(dim) + 1))
    dim = pd.concat([unknown_row, dim], ignore_index=True)
    log.info(f"Dim_Reason: {len(dim)} rows (incl. N/A placeholder)")
    return dim


def build_dim_date(df):
    start      = df['Date'].min().normalize()
    end        = df['Date'].max().normalize()
    date_range = pd.date_range(start=start, end=end)
    dim = pd.DataFrame({
        'Date_Key':   date_range.strftime('%Y%m%d').astype(int),
        'Full_Date':  date_range,
        'Day':        date_range.day,
        'Day_Name':   date_range.day_name(),
        'Month_Num':  date_range.month,
        'Month':      date_range.month_name(),
        'Quarter':    date_range.quarter,
        'Year':       date_range.year,
        'Is_Weekday': (date_range.weekday < 5).astype(int)
    })
    log.info(f"Dim_Date: {len(dim)} rows ({start.date()} → {end.date()})")
    return dim


# =============================================================================
# FACT TABLE
# =============================================================================

def build_fact_table(df, dim_agent, dim_satisfaction, dim_reason):
    fact = df.copy()

    # Y/N → 1/0
    fact = convert_yn_columns(fact)

    # Derive AHT — nulls stay null for unanswered calls naturally
    fact['AHT'] = fact['Talk Time (Secs)'] + fact['Hold Time (Secs)']

    # Map satisfaction labels to numeric score
    fact['Satisfaction_Score'] = fact['Satisfaction rating'].map(SATISFACTION_MAP)

    # Date surrogate key
    fact['Date_Key'] = fact['Date'].dt.strftime('%Y%m%d').astype(int)

    # Fill nulls before merge so they land on the N/A placeholder (ID = 0)
    fact['Satisfaction rating'] = fact['Satisfaction rating'].fillna('N/A — Call Not Answered')
    fact['Main Call Reason']    = fact['Main Call Reason'].fillna('N/A — Call Not Answered')
    fact['Sub Call Reason']     = fact['Sub Call Reason'].fillna('N/A — Call Not Answered')

    # Map surrogate keys — drop_duplicates on join key prevents row multiplication
    fact = fact.merge(
        dim_agent[['Agent_ID', 'Agent']].drop_duplicates('Agent'),
        on='Agent', how='left'
    )
    fact = fact.merge(
        dim_satisfaction[['Satisfaction_rating_ID', 'Satisfaction rating']].drop_duplicates('Satisfaction rating'),
        on='Satisfaction rating', how='left'
    )
    fact = fact.merge(
        dim_reason[['Reason_ID', 'Main Call Reason', 'Sub Call Reason']].drop_duplicates(['Main Call Reason', 'Sub Call Reason']),
        on=['Main Call Reason', 'Sub Call Reason'], how='left'
    )

    # Final column selection
    fact = fact[[
        'Call Id',
        'Date_Key',
        'Agent_ID',
        'Answered (Y/N)',
        'Issue Resolved?',
        'Resolved within SLA?',
        'First Call Resolution',
        'Speed of Answer (Secs)',
        'Talk Time (Secs)',
        'Hold Time (Secs)',
        'AHT',
        'Satisfaction_Score',
        'Satisfaction_rating_ID',
        'Reason_ID'
    ]]

    return fact


# =============================================================================
# KPI SUMMARY
# =============================================================================

def build_kpi_summary(fact):
    answered = fact[fact['Answered (Y/N)'] == 1]
    summary  = pd.DataFrame([{
        'Total Calls':                len(fact),
        'Answered Calls':             len(answered),
        'Unanswered Calls':           len(fact) - len(answered),
        'Answer Rate %':              round(len(answered) / len(fact) * 100, 2),
        'Avg AHT (Secs)':             round(answered['AHT'].mean(), 2),
        'Avg Speed of Answer (Secs)': round(answered['Speed of Answer (Secs)'].mean(), 2),
        'Avg Talk Time (Secs)':       round(answered['Talk Time (Secs)'].mean(), 2),
        'Avg Hold Time (Secs)':       round(answered['Hold Time (Secs)'].mean(), 2),
        'Avg Satisfaction Score':     round(answered['Satisfaction_Score'].mean(), 2),
        'FCR Rate %':                 round(answered['First Call Resolution'].mean() * 100, 2),
        'Issue Resolution Rate %':    round(answered['Issue Resolved?'].mean() * 100, 2),
        'SLA Compliance Rate %':      round(answered['Resolved within SLA?'].mean() * 100, 2),
    }])
    log.info("KPI_Summary: built successfully")
    return summary


# =============================================================================
# STYLING
# =============================================================================

def style_sheet(ws, header_color="1F4E79"):
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    from openpyxl.utils import get_column_letter

    header_fill = PatternFill("solid", fgColor=header_color)
    header_font = Font(name="Arial", bold=True, color="FFFFFF", size=10)
    body_font   = Font(name="Arial", size=10)
    center      = Alignment(horizontal="center", vertical="center")
    left        = Alignment(horizontal="left",   vertical="center")
    thin        = Side(style="thin", color="D9D9D9")
    border      = Border(left=thin, right=thin, top=thin, bottom=thin)

    for cell in ws[1]:
        cell.fill      = header_fill
        cell.font      = header_font
        cell.alignment = center
        cell.border    = border

    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.font      = body_font
            cell.border    = border
            cell.alignment = center if isinstance(cell.value, (int, float)) else left

    for col_idx in range(1, ws.max_column + 1):
        col_letter = get_column_letter(col_idx)
        max_len = max(
            (len(str(ws.cell(row=r, column=col_idx).value or "")) for r in range(1, ws.max_row + 1)),
            default=10
        )
        ws.column_dimensions[col_letter].width = min(max_len + 4, 40)

    ws.freeze_panes    = "A2"
    ws.auto_filter.ref = ws.dimensions


# =============================================================================
# MAIN ETL
# =============================================================================

def run_etl(input_xlsx="Call-Center-Dataset.xlsx", output_xlsx="CallCenter_Warehouse_Output.xlsx"):
    log.info("=" * 55)
    log.info("CALL CENTER ETL — START")
    log.info("=" * 55)

    # Delete existing output file if found
    if os.path.exists(output_xlsx):
        os.remove(output_xlsx)
        log.info(f"Deleted existing file: {output_xlsx}")

    # --- EXTRACT ---
    log.info(f"Reading: {input_xlsx}")
    df = pd.read_excel(input_xlsx)
    df['Date'] = pd.to_datetime(df['Date'])
    validate_input(df)

    # --- TRANSFORM ---
    log.info("Building dimension tables...")
    dim_agent_group  = build_dim_agent_group(df)
    dim_agent        = build_dim_agent(df, dim_agent_group)
    dim_satisfaction = build_dim_satisfaction(df)
    dim_reason       = build_dim_reason(df)
    dim_date         = build_dim_date(df)

    log.info("Checking dimension key uniqueness...")
    check_dim_uniqueness(dim_agent, dim_satisfaction, dim_reason)

    log.info("Building fact table...")
    fact = build_fact_table(df, dim_agent, dim_satisfaction, dim_reason)
    validate_fact_table(fact, df)

    log.info("Building KPI summary...")
    kpi_summary = build_kpi_summary(fact)

    # --- LOAD ---
    log.info(f"Writing to: {output_xlsx}")
    sheet_configs = [
        ("Fact_Table",       fact,             "1F4E79"),
        ("Dim_Agent",        dim_agent,        "375623"),
        ("Dim_Agent_Group",  dim_agent_group,  "1D6A4A"),
        ("Dim_Satisfaction", dim_satisfaction, "7B3F00"),
        ("Dim_Reason",       dim_reason,       "4A235A"),
        ("Dim_Date",         dim_date,         "154360"),
        ("KPI_Summary",      kpi_summary,      "8B0000"),
    ]

    with pd.ExcelWriter(output_xlsx, engine='openpyxl') as writer:
        for sheet_name, data, color in sheet_configs:
            data.to_excel(writer, sheet_name=sheet_name, index=False)
            style_sheet(writer.sheets[sheet_name], header_color=color)
            log.info(f"  ✓ {sheet_name} ({len(data):,} rows)")

    log.info("=" * 55)
    log.info(f"ETL COMPLETE → {output_xlsx}")
    log.info("=" * 55)


if __name__ == "__main__":
    run_etl()

2026-04-12 03:33:44,031 [INFO] =======================================================
2026-04-12 03:33:44,033 [INFO] CALL CENTER ETL — START
2026-04-12 03:33:44,034 [INFO] =======================================================
2026-04-12 03:33:44,036 [INFO] Deleted existing file: CallCenter_Warehouse_Output.xlsx
2026-04-12 03:33:44,038 [INFO] Reading: Call-Center-Dataset.xlsx
2026-04-12 03:33:45,073 [INFO] Input validated — 5,000 rows, 15 columns
2026-04-12 03:33:45,074 [INFO] Building dimension tables...
2026-04-12 03:33:45,077 [INFO] Dim_Agent_Group: 5 rows
2026-04-12 03:33:45,083 [INFO] Dim_Agent: 40 rows
2026-04-12 03:33:45,093 [INFO] Dim_Satisfaction: 6 rows (incl. N/A placeholder)
2026-04-12 03:33:45,104 [INFO] Dim_Reason: 16 rows (incl. N/A placeholder)
2026-04-12 03:33:45,109 [INFO] Dim_Date: 91 rows (2024-01-01 → 2024-03-31)
2026-04-12 03:33:45,110 [INFO] Checking dimension key uniqueness...
2026-04-12 03:33:45,111 [WARNING] Dim_Agent: 32 duplicate keys on 'Agent' — will cau

In [7]:
df = pd.read_excel('Call-Center-Dataset.xlsx')
print(df['Satisfaction rating'].unique())

[nan 'Extremely Satisfied' 'Neutral' 'Satisfied' 'Extremely Dissatisfied'
 'Dissatisfied']
